# Olist Brazilian E-Commerce: RFM Segmentation and Cohort Retention

Olist is a Brazilian marketplace. Its public dataset covers roughly 100,000 orders across 93,000 unique customers from 2016 through 2018, spread over nine CSV tables that join into a clean star schema around `orders`. This notebook teaches two techniques that operators use to read customer data: RFM segmentation and cohort retention. The punchline sits inside the data itself. Only 3 percent of Olist customers ever place a second order, so the cohort matrix is mostly empty and the RFM "frequency" column barely moves. That shape is the finding. Everything else in the notebook is how to see it.

## 1. Load the nine CSVs and resolve paths

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Kaggle-safe paths: read from the competition mount when present,
# otherwise fall back to a local `data/` folder.
KAGGLE = Path('/kaggle/input/brazilian-ecommerce')
LOCAL = Path('data')
ROOT = KAGGLE if KAGGLE.exists() else LOCAL

def load(name: str) -> pd.DataFrame:
    return pd.read_csv(ROOT / name)

orders    = load('olist_orders_dataset.csv')
customers = load('olist_customers_dataset.csv')
items     = load('olist_order_items_dataset.csv')
reviews   = load('olist_order_reviews_dataset.csv')

print(f'orders:    {len(orders):>8,}')
print(f'customers: {len(customers):>8,}')
print(f'items:     {len(items):>8,}')
print(f'reviews:   {len(reviews):>8,}')

Nine tables ship with the dataset; this notebook uses four of them directly. `orders` carries one row per order with status and timestamps. `customers` maps the per-order `customer_id` to a stable `customer_unique_id` that survives across orders. `order_items` is the line-item table with prices and freight. `reviews` is post-delivery star ratings.

The `customer_id` vs. `customer_unique_id` distinction is the first gotcha. A returning customer gets a new `customer_id` on each order, so aggregating on that column would make every customer look new. The retention analysis lives or dies on switching to `customer_unique_id` before grouping.

## 2. Filter to delivered orders and join line-item totals

In [ ]:
# Parse timestamps.
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

# Drop cancelled/unavailable orders -- only delivered orders carry completed revenue.
delivered = orders.loc[orders['order_status'] == 'delivered'].copy()
print(f'delivered: {len(delivered):,} of {len(orders):,}')

# Roll line items up to order totals. One order can have multiple items.
order_totals = (items.groupby('order_id')
                     .agg(items_count=('order_item_id', 'max'),
                          items_value=('price', 'sum'),
                          freight=('freight_value', 'sum'))
                     .reset_index())
order_totals['order_total'] = order_totals['items_value'] + order_totals['freight']

# Three-way join: delivered orders -> customer -> order total.
full = (delivered
        .merge(customers, on='customer_id', how='left')
        .merge(order_totals, on='order_id', how='left')
        .dropna(subset=['order_total', 'order_purchase_timestamp']))

full['purchase_month'] = full['order_purchase_timestamp'].dt.to_period('M').dt.to_timestamp()
print(f'full join: {len(full):,} rows')

The join chain is the analytics equivalent of a star schema lookup. `orders` is the fact table. `customers` is a degenerate dimension that gives each order its unique customer. `order_items` is a child fact that aggregates up to the order grain. After the merge, every row has one delivered order, its customer, and its gross value in Brazilian reais. Total revenue across the window lands near R$ 15.4M.

## 3. The headline numbers

In [ ]:
orders_per_customer = full.groupby('customer_unique_id')['order_id'].nunique()
summary = {
    'unique customers':        orders_per_customer.size,
    'delivered orders':        len(full),
    'median order (BRL)':      round(full['order_total'].median(), 2),
    'mean order (BRL)':        round(full['order_total'].mean(), 2),
    'total revenue (BRL)':     round(full['order_total'].sum(), 0),
    'single-order share':      round((orders_per_customer == 1).mean(), 4),
    'repeat-customer share':   round((orders_per_customer >= 2).mean(), 4),
}
pd.Series(summary).to_frame('value')

Median order is R$ 105 on a mean of R$ 160 — a right-skewed distribution typical of marketplaces that mix small consumables with occasional appliance and furniture purchases. The single-order share is 97 percent. That single number carries most of the notebook's weight.

## 4. Monthly revenue

In [ ]:
monthly = (full.groupby('purchase_month')
               .agg(revenue=('order_total', 'sum'),
                    orders=('order_id', 'count')).reset_index())

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(monthly['purchase_month'], monthly['revenue'] / 1000,
        'o-', lw=2.2, color='#166534')
ax.fill_between(monthly['purchase_month'], 0, monthly['revenue'] / 1000,
                color='#fbbf24', alpha=0.35)
ax.set_xlabel('Month'); ax.set_ylabel('Revenue (thousand BRL)')
ax.set_title('Monthly revenue: Black Friday 2017 peak, stable 2018')
fig.autofmt_xdate(); plt.show()

Revenue ramps from near zero in late 2016 through the first sustained growth in Q1 2017, peaks during the Brazilian Black Friday window in November 2017, then settles around R$ 1.1M per month for most of 2018. The last observation in the data cuts off mid-month, so anyone reading the last bar should treat it as partial.

## 5. Method primer: what RFM actually is

RFM segments customers along three axes and scores each one from 1 to 5.

**Recency** is the number of days since the customer's last order as of a chosen snapshot date. Recent customers get high R scores. Dormant customers get low ones.

**Frequency** counts how many distinct orders the customer has placed over the observation window. Loyal repeat buyers get high F scores; one-and-done buyers get 1s.

**Monetary** sums the customer's total spend. Big-ticket customers get high M scores regardless of whether they return.

The standard scoring move is **quintile binning on the rank**. Rank every customer on each of the three dimensions, then split the rank distribution into five equal-sized buckets. Customers in the top quintile score 5; customers in the bottom quintile score 1. Quintile binning on the *rank* (instead of on the raw value) dodges the problem that most customers tie at frequency = 1, which would otherwise leave `qcut` unable to produce five distinct bins. The three scores then combine into named segments like "Champions" (recent + frequent + high-value), "At risk" (was frequent, now dormant), and "Lost" (dormant and low-value).

RFM is a descriptive segmentation, not a predictive model. Its usefulness is that marketing and retention teams can take action on the buckets — a welcome series for New customers, a win-back campaign for At-risk, a loyalty tier for Champions — without needing a classifier or a propensity model on top.

## 6. Compute the RFM table

In [ ]:
snapshot = full['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = (full.groupby('customer_unique_id')
           .agg(recency=('order_purchase_timestamp', lambda s: (snapshot - s.max()).days),
                frequency=('order_id', 'nunique'),
                monetary=('order_total', 'sum'))
           .reset_index())

# Quintile-score each dimension using rank to tolerate ties.
rfm['r_score'] = pd.qcut(rfm['recency'].rank(method='first'), 5, labels=[5,4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['rfm_total'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

rfm.head()

Recency is inverted when scoring: lower days-since-last-order gets the higher score, because recent is better. Frequency and Monetary run the other way — more is better. `rfm_total` is the simple sum of the three scores, useful as a single-number stand-in when slicing.

## 7. Bucket customers into named segments

In [ ]:
def segment(row):
    r, f, m = row.r_score, row.f_score, row.m_score
    if r >= 4 and f >= 4: return 'Champions'
    if r >= 4 and f <= 2: return 'New / Recent'
    if r >= 3 and m >= 4: return 'Big spenders'
    if r <= 2 and f >= 4: return 'At risk'
    if r <= 2 and f <= 2: return 'Lost'
    return 'Others'

rfm['segment'] = rfm.apply(segment, axis=1)
seg_counts = rfm['segment'].value_counts()
seg_rev    = rfm.groupby('segment')['monetary'].sum().reindex(seg_counts.index)

out = pd.DataFrame({
    'customers': seg_counts,
    'revenue_brl': seg_rev.round(0).astype(int),
    'revenue_per_customer': (seg_rev / seg_counts).round(2),
})
out.sort_values('revenue_brl', ascending=False)

Big spenders is the smallest segment at about 10,000 customers, but pulls the highest revenue total at R$ 292 per customer. Champions are the closest to loyal, recent, and valuable — the segment worth protecting. At risk sits between Champions and Lost; customers in this bucket used to buy frequently and have gone quiet, which is the canonical retention-campaign target.

## 8. Plot customer count and revenue per segment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
palette = ['#166534', '#fbbf24', '#1e40af', '#bef264', '#6b7280', '#fde047']
axes[0].bar(seg_counts.index, seg_counts.values, color=palette[:len(seg_counts)])
axes[0].set_title('Customers per RFM segment'); axes[0].tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Customers')
axes[1].bar(seg_rev.index, seg_rev.values / 1000, color=palette[:len(seg_rev)])
axes[1].set_title('Revenue per RFM segment (thousand BRL)'); axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylabel('Revenue (thousand BRL)')
plt.tight_layout(); plt.show()

Two things stand out. The "Others" bucket is the largest by count but the smallest by revenue per customer — these are the ambient, low-value, low-frequency customers that every marketplace accumulates. And the revenue bars are roughly flat across Lost, New / Recent, At risk, and Champions, which means revenue concentration in this dataset is not as strong as it would be for a subscription business; instead, it is spread across the base because the base mostly buys once.

## 9. Method primer: cohort retention

A cohort is a group of customers defined by a shared start event — here, the month of each customer's first delivered order. Retention is the share of a cohort that orders again in any later period.

The key word is *conditional*. Cohort retention is always expressed as a percentage of the original cohort size, not of the full customer base. A 3 percent retention in month 6 for the March 2017 cohort means that of the customers who first ordered in March 2017, only 3 percent also ordered in September 2017. Month 0 is 100 percent by construction — every customer in the cohort is, by definition, active in the month they first ordered.

The retention matrix is triangular. Older cohorts have many months of follow-up (12 or more). Younger cohorts have only a few. Reading the matrix means comparing *across rows for the same column*: does the September 2017 cohort retain better at month 3 than the May 2017 cohort did at its month 3? If so, acquisition quality improved. If every row looks identical at month 3, the acquisition mix is stable.

For Olist, every non-month-0 cell sits below 5 percent and most below 2. Retention isn't decaying; it collapses immediately. That is a signal about the business, not about the measurement. Olist's catalogue is dominated by one-shot purchases — appliances, furniture, electronics — so the customer's effective lifetime in the dataset is a single order plus a narrow tail.

## 10. Build the cohort retention matrix

In [ ]:
# Assign every order a cohort (customer's first-purchase month) and a period
# (the month the order itself was placed).
first = (full.groupby('customer_unique_id')['order_purchase_timestamp']
             .min().dt.to_period('M'))
full['cohort'] = full['customer_unique_id'].map(first)
full['period'] = full['order_purchase_timestamp'].dt.to_period('M')

# Month offset: 0 for the cohort's own month, 1 for next month, etc.
full['cohort_index'] = ((full['period'].dt.year  - full['cohort'].dt.year) * 12
                      + (full['period'].dt.month - full['cohort'].dt.month))

# Unique customers active per (cohort, offset) cell.
cohort = (full.groupby(['cohort', 'cohort_index'])['customer_unique_id']
              .nunique().unstack(fill_value=0))

sizes = cohort[0]  # month-0 counts = original cohort sizes
retention = cohort.divide(sizes, axis=0)

# Keep cohorts with at least 100 customers, first 12 months only.
retention = retention.loc[sizes >= 100].iloc[:, :12]
(retention * 100).round(2).head()

Dividing each cell by the corresponding month-0 size is the step that turns raw counts into retention percentages. Without that normalisation, bigger cohorts would always dominate smaller ones in the heatmap, regardless of retention quality.

## 11. Cohort retention heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(retention * 100, cmap='YlGn', vmin=0, vmax=8,
            cbar_kws={'label': 'Retention (%)'},
            yticklabels=[str(p) for p in retention.index], ax=ax)
ax.set_xlabel('Months since first purchase')
ax.set_ylabel('Cohort (first-purchase month)')
ax.set_title('Olist cohort retention -- almost every post-month-0 cell under 3 percent')
plt.tight_layout(); plt.show()

Month 0 is the dark left column. Every row drops to the lightest shade immediately in month 1 and stays there. The better cohorts reach 4-5 percent retention at month 1 before falling back under 2 percent by month 3. No cohort recovers. In a healthy subscription or consumables business, this matrix would have a visible gradient that slowly fades from left to right; in Olist's marketplace data, the gradient is a cliff.

## 12. Delivery speed vs. review score

In [ ]:
j = reviews.merge(
    full[['order_id','order_purchase_timestamp','order_delivered_customer_date']],
    on='order_id', how='inner')
j['delivery_days'] = (j['order_delivered_customer_date']
                      - j['order_purchase_timestamp']).dt.days
j = j.loc[j['delivery_days'].between(0, 45)]
buckets = pd.cut(j['delivery_days'], bins=[-0.5, 3, 7, 14, 21, 30, 45])
score = j.groupby(buckets, observed=True)['review_score'].mean()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(['0-3','4-7','8-14','15-21','22-30','31-45'], score.values, color='#fbbf24', edgecolor='#166534')
ax.axhline(3.0, color='#1e40af', linestyle='--', lw=1)
ax.set_ylim(0, 5); ax.set_ylabel('Mean review score')
ax.set_xlabel('Delivery time bucket (days)')
ax.set_title('Delivery speed is the dominant review-score driver')
plt.tight_layout(); plt.show()

0-3 day delivery averages a 4.5-star review. Past 30 days, the mean falls below 3.0 — the threshold where customers stop recommending a retailer. Brazilian geography makes 40-day delivery times a real part of this dataset's distribution, and the cost shows up directly in the review column. For a marketplace operator, the operational finding is that shipping-time variance is a review-score risk before it is a logistics cost.

## 13. Findings

The dataset holds 96,478 delivered orders across 93,358 unique customers. Median order is R$ 105; total revenue across the two-year window is R$ 15.4M. The repeat-customer rate is 3 percent, so first-order revenue accounts for 97 percent of the total. RFM segmentation groups customers into six buckets that each carry comparable revenue, but "Big spenders" earns R$ 292 per customer against R$ 101 for "Others". Cohort retention collapses to below 3 percent after month 0 for every cohort. Delivery-time buckets map almost linearly to mean review score, with the 3.0 recommend threshold crossed past 30-day delivery.

## 14. Try this next

Fork this kernel and extend it in one of three directions. First, rebuild the cohort matrix on a *second-purchase* basis to ask how quickly the tiny repeat-customer population returns. Second, merge in the `products` and `product_category_name_translation` tables and recompute retention by category — furniture and appliances should show the lowest retention, while health_beauty and housewares may show higher. Third, score customers by RFM quintile directly and check whether segment membership in the first observed month predicts segment membership three months later. That last one starts to look like a predictive model rather than a descriptive one, and it is the natural bridge from this notebook into a customer-lifetime-value exercise.